In [ ]:
import os

import torch
import yaml
from dotenv import load_dotenv
from google import genai
from google.genai import types
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, PeftModel, TaskType, get_peft_model

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

dict_lora_path = {
    "c": "../data/lora/trained/peft_trainver2/lora_c_lora",
    "python": "../data/lora/trained/peft_trainver2/lora_python_lora",
    "html": "../data/lora/trained/peft_trainver2/lora_html_lora",
}


In [ ]:
dict_question_exp = {
    "Q1": "うどんのおすすめのトッピングを教えて。",
    "Q2": "桜は綺麗だね。",
    "Q3": "とりあえずLoRAやプロンプトをいじってみたけど、本末転倒かな？",
}

In [ ]:
peft_config = LoraConfig(
    r=8,   # LoRAのランク。学習の規模みたいなもの
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"], # 対象レイヤー
    lora_dropout=0.05, 
    bias="none",
    task_type=TaskType.CAUSAL_LM   # 因果言語モデルを指定
)

In [ ]:
with open("config/character_config.yaml", "r", encoding="utf-8") as file:
    character_config = yaml.safe_load(file)


## 1. 軽量モデル＋LoRA

### 1.1 設定

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    torch_dtype="auto", 
    device_map="auto"
)

In [ ]:
# 学習済みのLoRAを読み込む前のベースアダプターを作成

model = get_peft_model(base_model, peft_config, adapter_name="adapter_base")

In [ ]:
# 各キャラクターのLoRAのアダプターを作成

model.load_adapter(dict_lora_path["c"], adapter_name="adapter_c")
model.load_adapter(dict_lora_path["python"], adapter_name="adapter_python")
model.load_adapter(dict_lora_path["html"], adapter_name="adapter_html")


### 1.2 LoRAを読み込んでの応答

In [ ]:
def ask_llm_with_RoLA(model, tokenizer, temperature, adapter_name, system_text, text):

    model.set_adapter(adapter_name)
    
    messages = [
        {"role": "system", "content": f"{system_text}"},
        {"role": "user", "content": f"{text}"}
    ]

    # QwenシリーズのInstructモデル用に、チャット形式のテンプレートを適用
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    prompt = tokenizer([text], return_tensors="pt").to(model.device)

    # TODO
    # 生成部分を関数化して効率的に使い回しできる。

    # 推論だけしたいので、勾配計算と保存をしない設定
    with torch.no_grad():
        generated_ids = model.generate(
            **prompt,
            max_new_tokens=512,  # 生成する最大トークン数
            do_sample=False,
            # temperature=temperature,
            # top_p=0.9
        )
    
    # 入力部分を削り、LLMが生成した部分だけを取り出す
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(prompt.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return response

In [ ]:
temperature = 0.1 #　動作検証段階なので、回答がある程度揃うように下げる
# text = "こんにちは！うどんのおすすめのトッピングは？"

text = dict_question_exp["Q3"]

character = "html"
adapter_name = f"adapter_{character}"

print(text,"\n")
print(adapter_name)

system_text = character_config[character]["personality"]

response = ask_llm_with_RoLA(
    model=model,
    tokenizer=tokenizer,
    temperature=temperature,
    adapter_name=adapter_name,
    system_text=system_text,
    text=text
)

In [ ]:
print(response)

## 2. 軽量モデル＋プロンプトエンジニアリング

### 2.1 ベースモデル

In [ ]:
def ask_llm_with_original_model(
    model, tokenizer, temperature, character_config, character_name, text
):

    print(
        character_config[character_name]["personality"]
    )

    system_text =  f"あなたは、{character_config[character_name]["personality"]}、というキャラクターです。キャラクターになりきって、返答してください"

    messages = [
        {"role": "system", "content": system_text},
        {"role": "user", "content": f"{text}"}
    ]

    # QwenシリーズのInstructモデル用に、チャット形式のテンプレートを適用
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    prompt = tokenizer([text], return_tensors="pt").to(model.device)

    # アダプターをオフにする
    # 推論だけしたいので、勾配計算と保存をしない設定
    with model.disable_adapter():
        with torch.no_grad():
            generated_ids = model.generate(
                **prompt,
                max_new_tokens=512,  # 生成する最大トークン数
                do_sample=False,
                # temperature=temperature,
                # top_p=0.9
            )
    
    # 入力部分を削り、LLMが生成した部分だけを取り出す
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(prompt.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return response

In [ ]:
temperature = 0.1 #　動作検証段階なので、回答がある程度揃うように下げる
text = "こんにちは！うどんのおすすめのトッピングは？"

text = dict_question_exp["Q3"]

character = "html"

print(text,"\n")
print(character)

response = ask_llm_with_original_model(
    model=model,
    tokenizer=tokenizer,
    temperature=temperature,
    character_config=character_config,
    character_name=character,
    text=text
)

In [ ]:
print(response)

### 2.2 簡易的なプロンプトエンジニアリング

In [ ]:
def ask_llm_with_prompt_engineering(
    model, tokenizer, temperature, character_config, character_name, text
):


    print(f"キャラクター：{character_config[character_name]["name"]}\n")

    system_prompt = f"""
        あなたは、{character_config[character_name]["personality"]}
        この設定で、ユーザーと会話をしてください。
    
        【返答のルール】
        - {character_config[character_name]["response_rules"]["rule1"]}
        - {character_config[character_name]["response_rules"]["rule2"]}
        - {character_config[character_name]["response_rules"]["rule3"]}
        - 会話のトーンは、{character_config[character_name]["tone"]}です。
        - 必ず、２行以内で返事を行ってください。
        - 後述する返答の例はあくまで例です。口調などは真似してください。ただし、例の文章をそのまま返答に入れることは禁止します。

        【返答の例】
        - {character_config[character_name]["few-shot"]["exp1"]}
        - {character_config[character_name]["few-shot"]["exp2"]}
        - {character_config[character_name]["few-shot"]["exp3"]}

    """

    print(f"{system_prompt}\n\n")

    messages = [
        {"role": "system", "content": f"{system_prompt}"},
        {"role": "user", "content": f"{text}"}
    ]

    # QwenシリーズのInstructモデル用に、チャット形式のテンプレートを適用
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    prompt = tokenizer([text], return_tensors="pt").to(model.device)

    # アダプターをオフにする
    # 推論だけしたいので、勾配計算と保存をしない設定
    with model.disable_adapter():
        with torch.no_grad():
            generated_ids = model.generate(
                **prompt,
                max_new_tokens=512,  # 生成する最大トークン数
                do_sample=False,
                # temperature=temperature,
                # top_p=0.9
            )
    
    # 入力部分を削り、LLMが生成した部分だけを取り出す
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(prompt.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]


    return response

In [ ]:
temperature = 0.1 #　動作検証段階なので、回答がある程度揃うように下げる
# text = "こんにちは！ラーメンとうどん、どっちがいいですか？"

text = dict_question_exp["Q3"]
print(text,"\n")

character = "html"

response = ask_llm_with_prompt_engineering(
    model=model, 
    tokenizer=tokenizer, 
    temperature=temperature,
    character_config=character_config, 
    character_name=character, 
    text=text
)

In [ ]:
print(response)

## 3. LoRA + 簡易的なプロンプトエンジニアリング

In [ ]:
def ask_llm_with_LoRA_and_prompt_engineering(
    model, tokenizer, temperature, adapter_name, character_config, character_name, text
):

    print(f"キャラクター：{character_config[character_name]["name"]}\n")

    # TODO
    # system prompoを作成する関数を作って、効率化する。
    system_prompt = f"""
        あなたは、{character_config[character_name]["personality"]}
        この設定で、ユーザーと会話をしてください。
    
        【返答のルール】
        - {character_config[character_name]["response_rules"]["rule1"]}
        - {character_config[character_name]["response_rules"]["rule2"]}
        - {character_config[character_name]["response_rules"]["rule3"]}
        - 会話のトーンは、{character_config[character_name]["tone"]}です。
        - 必ず、２行以内で返事を行ってください。
        - 後述する返答の例はあくまで例です。口調などは真似してください。ただし、例の文章をそのまま返答に入れることは禁止します。

        【返答の例】
        - {character_config[character_name]["few-shot"]["exp1"]}
        - {character_config[character_name]["few-shot"]["exp2"]}
        - {character_config[character_name]["few-shot"]["exp3"]}

    """

    print(f"{system_prompt}\n\n")


    # LoRAのadaperを指定
    model.set_adapter(adapter_name)

    messages = [
        {"role": "system", "content": f"{system_prompt}"},
        {"role": "user", "content": f"{text}"}
    ]

    # QwenシリーズのInstructモデル用に、チャット形式のテンプレートを適用
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    prompt = tokenizer([text], return_tensors="pt").to(model.device)

    # 推論だけしたいので、勾配計算と保存をしない設定
    with torch.no_grad():
        generated_ids = model.generate(
            **prompt,
            max_new_tokens=512,  # 生成する最大トークン数
            do_sample=False,
            # temperature=temperature,      # 自由度（好みに応じて変更）
            # top_p=0.9
        )
    
    # 入力部分を削り、LLMが生成した部分だけを取り出す
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(prompt.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return response

In [ ]:
temperature = 0.1 #　動作検証段階なので、回答がある程度揃うように下げる

# text = "こんにちは！うどんのおすすめのトッピングは？"
text = dict_question_exp["Q3"]

character = "html"
adapter_name = f"adapter_{character}"

print(text,"\n")
print(adapter_name)

system_text = character_config[character]["personality"]

response = ask_llm_with_LoRA_and_prompt_engineering(
    model=model,
    tokenizer=tokenizer,
    temperature=temperature,
    adapter_name=adapter_name,
    character_config=character_config, 
    character_name=character, 
    text=text
)

In [ ]:
print(response)

## 4. gemini apiとの動作比較

In [ ]:
load_dotenv()
gemini_api_key = os.getenv("API_KEY")

In [ ]:
generation_config = {
    "temperature": 0.2,
    "max_output_tokens": 1500, 
}

model_gemini = "gemini-2.5-flash"

In [ ]:
client_gemini = genai.Client(api_key = gemini_api_key)

In [ ]:
temperature = 0.1 #　動作検証段階なので、回答がある程度揃うように下げる

contents = dict_question_exp["Q3"]

character_name = "html"

print(contents[:10])

# system_text = character_config[character]["personality"]

print(f"キャラクター：{character_config[character_name]["name"]}\n")

system_text = f"""
    あなたは、{character_config[character_name]["personality"]}
    この設定で、ユーザーと会話をしてください。

    【返答のルール】
    - {character_config[character_name]["response_rules"]["rule1"]}
    - {character_config[character_name]["response_rules"]["rule2"]}
    - {character_config[character_name]["response_rules"]["rule3"]}
    - 会話のトーンは、{character_config[character_name]["tone"]}です。
    - 必ず、２行以内で返事を行ってください。
    - 後述する返答の例はあくまで例です。口調などは真似してください。ただし、例の文章をそのまま返答に入れることは禁止します。

    【返答の例】
    - {character_config[character_name]["few-shot"]["exp1"]}
    - {character_config[character_name]["few-shot"]["exp2"]}
    - {character_config[character_name]["few-shot"]["exp3"]}

"""

response = client_gemini.models.generate_content(
    model=model_gemini,
    contents=contents,
    config=types.GenerateContentConfig(
        system_instruction=system_text,
        temperature=temperature
    )
)

In [ ]:
print(response.text)